# 2. Targeting CSRO from literature

Special quasi-random structures (α=0) are useful, but real alloys often have
**chemical short-range order** — measurably non-zero α values at specific
species pairs and shells, often determined by APT, neutron scattering, or DFT.

The upstream-extending `SRO` INPUT keyword in `scraps-perpair` lets you target
those values directly. This notebook reproduces a literature-style CSRO
pattern on NiCoCr and confirms the structure achieves it.


## A literature-style CSRO pattern

NiCoCr is the canonical CSRO testbed. A representative pattern from the
literature is:

- α(Ni-Cr) at NN ≈ −0.10 (Cr-Cr repulsion drives Ni-Cr to prefer adjacency)
- α(Co-Cr) at NN ≈ −0.05 (weaker preference)
- α(Ni-Co) at NN ≈ 0 (effectively random)

The sum-rule constraint $\sum_j c_j \alpha_{ij} = 0$ implies α(Cr-Cr) at NN
ends up positive (Cr-Cr clustering avoided), which we don't have to target
explicitly — it emerges.


In [1]:
import sys
from pathlib import Path
REPO = Path.cwd().parent
sys.path.insert(0, str(REPO / "tools"))

from ase.build import bulk
from scraps_ase import run_sqs_from_atoms

primitive = bulk("Cu", "fcc", a=3.56, cubic=True)

# (species_i, species_j, shell_index, alpha_target)
sro_targets = [
    (0, 2, 0, -0.10),  # Ni-Cr at NN
    (1, 2, 0, -0.05),  # Co-Cr at NN
    (0, 1, 0,  0.00),  # Ni-Co at NN
]

csro = run_sqs_from_atoms(
    primitive,
    species=["Ni", "Co", "Cr"],
    element_counts=[36, 36, 36],
    cell_dim=(3, 3, 3),
    max_shellnum=2,
    weights=[2.0, 1.0],   # weight NN twice as much as 2NN
    sro_targets=sro_targets,
    auto_budget=2,        # thorough MC budget; see notebook 1
    rng_seed=42,
    work_dir=Path.cwd() / "nb2_csro",
)
print("Final fitness:", csro.info["scraps_final_fitness"])


Final fitness: 1.78885


## Verify the achieved alphas

`verify_sro.py` can be pointed at the INPUT file via `--target` and report
the per-pair pass/fail against a tolerance — useful for CI gates.


In [2]:
import subprocess
verify = REPO / "tools" / "verify_sro.py"
work_dir = csro.info["scraps_work_dir"]
result = subprocess.run(
    [sys.executable, str(verify),
     str(Path(work_dir) / "SCRAPS.vasp"),
     "--species", "Ni,Co,Cr", "--shells", "2"],
    capture_output=True, text=True,
)
print(result.stdout)


Lattice: (10.68, 10.68, 10.68) (Angstrom), angles (90.0, 90.0, 90.0)
Atoms: 108 total, species ['Ni', 'Co', 'Cr']
Detected shells: 2
  shell 0: r ∈ [2.5173, 2.5173], z = 12
  shell 1: r ∈ [3.5600, 3.5600], z = 6

=== Shell 0 (z = 12) ===
                Ni       Co       Cr
    Ni  +0.0972 +0.0000 -0.0972 
    Co  +0.0000 +0.0556 -0.0556 
    Cr  -0.0972 -0.0556 +0.1528 

=== Shell 1 (z = 6) ===
                Ni       Co       Cr
    Ni  +0.0000 +0.0000 +0.0000 
    Co  +0.0000 +0.0000 +0.0000 
    Cr  +0.0000 +0.0000 +0.0000 




## Compare to a pure random SQS

The same composition with `sro_targets=[]` gives the SQS baseline. Notice how
α(Ni-Cr) at shell 0 drops to ~0 (random) while in the CSRO run it sits near
the −0.10 target.


In [3]:
sqs = run_sqs_from_atoms(
    primitive,
    species=["Ni", "Co", "Cr"],
    element_counts=[36, 36, 36],
    cell_dim=(3, 3, 3),
    max_shellnum=2,
    weights=[1, 1],
    auto_budget=2,
    rng_seed=42,
    work_dir=Path.cwd() / "nb2_sqs_baseline",
)
result = subprocess.run(
    [sys.executable, str(verify),
     str(Path(sqs.info["scraps_work_dir"]) / "SCRAPS.vasp"),
     "--species", "Ni,Co,Cr", "--shells", "2"],
    capture_output=True, text=True,
)
print(result.stdout)


Lattice: (10.68, 10.68, 10.68) (Angstrom), angles (90.0, 90.0, 90.0)
Atoms: 108 total, species ['Ni', 'Co', 'Cr']
Detected shells: 2
  shell 0: r ∈ [2.5173, 2.5173], z = 12
  shell 1: r ∈ [3.5600, 3.5600], z = 6

=== Shell 0 (z = 12) ===
                Ni       Co       Cr
    Ni  +0.0000 +0.0000 +0.0000 
    Co  +0.0000 +0.0000 +0.0000 
    Cr  +0.0000 +0.0000 +0.0000 

=== Shell 1 (z = 6) ===
                Ni       Co       Cr
    Ni  +0.0000 +0.0000 +0.0000 
    Co  +0.0000 +0.0000 +0.0000 
    Cr  +0.0000 +0.0000 +0.0000 




The contrast — same composition, same cell, different SRO signatures — is the
unique value of the per-pair SRO API. Upstream SCRAPS only supported a single
scalar α target applied to every species pair, so this kind of targeted
ordering wasn't expressible.

The classical-ordered tests in `tests/{b2,l12,l10,d03,l11,l21}_*` use the
same machinery with α targets at the maximum (±1, ±1/3) bounds.
